# 02 Baseline Models

**Goal:** Establish baseline performance with simple, fast models before reaching for anything complex. I build two Logistic Regression variants — one on raw word counts, one on TF-IDF — to set a reference point that any heavier model will have to beat. I evaluate on a held-out validation set (not training data) to get an honest read on performance.

In [6]:
%pip install scikit-learn

You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.9/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Load and preprocess

Load the data, fill in empty subjects/messages, merge subject + body into a single `text` column, and encode the label as 1 (spam) / 0 (ham).

In [7]:
import pandas as pd

df = pd.read_csv('../data/enron_spam_data.csv')

# Some emails have no subject or no body — treat those as empty strings
df['Subject'] = df['Subject'].fillna('')
df['Message'] = df['Message'].fillna('')

# Combine subject + body so the model sees the whole email as one text field
df['text'] = df['Subject'] + ' ' + df['Message']

# Models need numeric labels,so map spam: 1,ham: 0
df['label'] = (df['Spam/Ham'] == 'spam').astype(int)

df[['text', 'label']].head()

,text,label
0,christmas tree farm pictures,0
1,"vastar resources , inc . gary , production fro...",0
2,calpine daily gas nomination - calpine daily g...,0
3,re : issue fyi - see note below - already done...,0
4,meter 7268 nov allocation fyi .\n- - - - - - -...,0


## Remove duplicates first

Spam datasets tend to repeat the same email many times. If the same email lands in both train and test,the model is really being tested on something it already saw, which fakes a high score. So I drop duplicates **before** splitting, not after.

In [8]:
# Keep only one copy of each unique email
df_unique = df.drop_duplicates(subset='text').reset_index(drop=True)
print("Before:", len(df), "-> after removing duplicates:", len(df_unique))

Before: 33716 -> after removing duplicates: 30494


## Train / validation / test split

A 70/15/15 split on the de-duplicated data. 
`stratify` keeps the spam/ham ratio identical across all three sets so no split ends up lopsided.

In [9]:
from sklearn.model_selection import train_test_split

#First peel off the test set,then split the rest into train and validation.
train_val, test = train_test_split(
    df_unique,test_size= 0.15,random_state= 42,stratify = df_unique['label']
)
train, val = train_test_split(
    train_val,test_size= 0.176,random_state= 42,stratify = train_val['label']
)

print('Train:',len(train))
print('Value:',len(val))
print('Test:',len(test))

Train: 21357
Value: 4562
Test: 4575


## Model 1 — Baseline (bag-of-words + Logistic Regression)

`CountVectorizer` turns each email into word counts. `min_df=5` ignores words that show up in fewer than 5 emails(too rare to be useful).I fit the vectorizer on the training set only, then apply that same vocabulary to validation and test — fitting on val/test would leak information.

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

count = CountVectorizer(min_df=5)
X_train = count.fit_transform(train['text'])
X_val = count.transform(val['text'])

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, train['label'])

print('Baseline Model trained')

Baseline Model trained


In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

val_pred = baseline_model.predict(X_val)

print('Baseline(Validation)')
print('Accuracy :', round(accuracy_score(val['label'], val_pred), 4))
print('Precision:', round(precision_score(val['label'], val_pred), 4))
print('Recall   :', round(recall_score(val['label'], val_pred), 4))
print('F1       :', round(f1_score(val['label'], val_pred), 4))

Baseline(Validation)
Accuracy : 0.986
Precision: 0.9779
Recall   : 0.9931
F1       : 0.9854


## Model 2 — TF-IDF + Logistic Regression

Same classifier, but the features are TF-IDF weighted instead of raw counts. I change only the vectorizer,so whatever difference I see comes purely from the feature representation.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf_idf = TfidfVectorizer(min_df=5)
X_train_tf = tf_idf.fit_transform(train['text'])
X_val_tf = tf_idf.transform(val['text'])

model_tf = LogisticRegression(max_iter=1000)
model_tf.fit(X_train_tf, train['label'])

print('TF-IDF Model trained')

TF-IDF Model trained


In [13]:
val_pred_tf = model_tf.predict(X_val_tf)

print('TF-IDF (validation)')
print('Accuracy :', round(accuracy_score(val['label'], val_pred_tf), 4))
print('Precision:', round(precision_score(val['label'], val_pred_tf), 4))
print('Recall   :', round(recall_score(val['label'], val_pred_tf), 4))
print('F1       :', round(f1_score(val['label'], val_pred_tf), 4))

TF-IDF (validation)
Accuracy : 0.986
Precision: 0.9753
Recall   : 0.9959
F1       : 0.9855


## Test-set evaluation + inference time

Final check on the held-out test set,reusing the vectorizers already fit on train. I also time how long prediction takes per email — this matters for deployment, where the transformer's accuracy has to be weighed against its much higher inference cost.

In [14]:
import time

# Transform the test set with the SAME vectorizers fit on train (no refitting)
X_test = count.transform(test['text'])
X_test_tf = tf_idf.transform(test['text'])

# Report precision, recall, F1, and inference time per email
def report(name, model, X):
    start = time.perf_counter()
    pred = model.predict(X)
    elapsed = time.perf_counter() - start
    n = len(test)
    print(f"{name:9} | P {precision_score(test['label'], pred):.4f} | "
          f"R {recall_score(test['label'], pred):.4f} | "
          f"F1 {f1_score(test['label'], pred):.4f} | "
          f"{elapsed/n*1e6:.1f} us/email")

print("=== TEST SET RESULTS ===")
report('Baseline', baseline_model, X_test)
report('TF-IDF', model_tf, X_test_tf)

=== TEST SET RESULTS ===
Baseline  | P 0.9802 | R 0.9941 | F1 0.9871 | 1.2 us/email
TF-IDF    | P 0.9789 | R 0.9973 | F1 0.9880 | 0.4 us/email


## Error analysis

Beyond the metrics, I looked at *which* emails the baseline got wrong — false positives (ham predicted as spam) and false negatives (spam predicted as ham) — to understand where the model struggles.

In [15]:
# Predict on the test set with the baseline model
test_pred = baseline_model.predict(X_test)

# Find misclassified emails
test_results = test.copy()
test_results['pred'] = test_pred

false_positives = test_results[(test_results['label'] == 0) & (test_results['pred'] == 1)]
false_negatives = test_results[(test_results['label'] == 1) & (test_results['pred'] == 0)]

print(f"False positives (ham flagged as spam): {len(false_positives)}")
print(f"False negatives (spam missed): {len(false_negatives)}")

# Look at a few false positives — real emails wrongly sent to spam
print("\n--- Sample FALSE POSITIVES (ham wrongly flagged) ---")
for text in false_positives['text'].head(3):
    print(text[:200], "\n")

# Look at a few false negatives — spam that slipped through
print("--- Sample FALSE NEGATIVES (spam missed) ---")
for text in false_negatives['text'].head(3):
    print(text[:200], "\n")

False positives (ham flagged as spam): 44
False negatives (spam missed): 13

--- Sample FALSE POSITIVES (ham wrongly flagged) ---
you are on bloomberg - news section i am at home tonight - give me a call .
~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ disclaimer ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~ ~
this e - mail is private and  

geir ' s goals ) 

fw : just some info before you vote mike triem
na ipaq ' eric geiger ' ; ' jacob
crawford ' ; ' laura gambrell ' ; ' lgambrell @ sammons - parker . com ' ; ' steve ' ;
' victor mamich '
cc : ' bg _ jg 

--- Sample FALSE NEGATIVES (spam missed) ---
luca ricci new years eve . . . luca ricci open dates :
2004 - december nye
2005 - january , february march
luca ricci - bio
the talented up and coming italian dj producer , was born and raised on a sm 

an information an i n f o r m a t i o
n
in the age of
information
finding quicker solutions and solving
problems quicker
when you are fully booked and
overloaded
with work there are still the possi

## Multi-seed validation

A single split can be lucky or unlucky.To check how stable the scores really are, I re-run both models across 5 different seeds and report mean ± standard deviation. This tells me whether the gap between models is real or just noise from one particular split.

In [16]:
import numpy as np

seeds = [0, 1, 2, 3, 4]
# Collect (accuracy, precision, recall, f1) for each model, one row per seed
baseline_scores, tfidf_scores = [], []

for seed in seeds:
    seed_train_val, seed_test = train_test_split(
        df_unique, test_size=0.15, random_state=seed, stratify=df_unique['label'])
    seed_train, seed_val = train_test_split(
        seed_train_val, test_size=0.176, random_state=seed, stratify=seed_train_val['label'])

    for vectorizer_class, scores in [(CountVectorizer, baseline_scores),
                                     (TfidfVectorizer, tfidf_scores)]:
        vectorizer = vectorizer_class(min_df=5)
        X_tr = vectorizer.fit_transform(seed_train['text'])
        X_te = vectorizer.transform(seed_test['text'])
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_tr, seed_train['label'])
        pred = clf.predict(X_te)
        scores.append((
            accuracy_score(seed_test['label'], pred),
            precision_score(seed_test['label'], pred),
            recall_score(seed_test['label'], pred),
            f1_score(seed_test['label'], pred)
        ))

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
summary = {}
for name, scores in [('Baseline (Count + LR)', baseline_scores),
                     ('TF-IDF + LR', tfidf_scores)]:
    arr = np.array(scores)
    summary[name] = {metric_names[i]: f"{arr[:, i].mean():.4f} ± {arr[:, i].std():.4f}"
        for i in range(4)
    }

pd.DataFrame(summary).T

,Accuracy,Precision,Recall,F1
Baseline (Count + LR),0.9867 ± 0.0009,0.9814 ± 0.0010,0.9909 ± 0.0013,0.9861 ± 0.0010
TF-IDF + LR,0.9855 ± 0.0019,0.9747 ± 0.0039,0.9955 ± 0.0006,0.9850 ± 0.0019


## Final comparison (All three models)

Multi-seed mean ± std.The LR rows come from the cell above (5 seeds);
The DistilBERT row comes from notebook 03 (3 seeds). 
I show accuracy/precision/recall/F1 together because F1 alone would hide that the models differ on precision — the metric that matters most for spam.

In [17]:
results = {
    'Baseline (Count + LogReg)': {
        'Accuracy':  '0.9867 ± 0.0009',
        'Precision': '0.9814 ± 0.0010',
        'Recall':    '0.9909 ± 0.0013',
        'F1':        '0.9861 ± 0.0010',
    },
    'TF-IDF + LogReg': {
        'Accuracy':  '0.9855 ± 0.0019',
        'Precision': '0.9747 ± 0.0039',
        'Recall':    '0.9955 ± 0.0006',
        'F1':        '0.9850 ± 0.0019',
    },
    'DistilBERT (Transformer)': {
        'Accuracy':  '0.9941 ± 0.0005',
        'Precision': '0.9918 ± 0.0007',
        'Recall':    '0.9936 ± 0.0011',
        'F1':        '0.9927 ± 0.0005',
    },
}

pd.DataFrame(results).T

,Accuracy,Precision,Recall,F1
Baseline (Count + LogReg),0.9867 ± 0.0009,0.9814 ± 0.0010,0.9909 ± 0.0013,0.9861 ± 0.0010
TF-IDF + LogReg,0.9855 ± 0.0019,0.9747 ± 0.0039,0.9955 ± 0.0006,0.9850 ± 0.0019
DistilBERT (Transformer),0.9941 ± 0.0005,0.9918 ± 0.0007,0.9936 ± 0.0011,0.9927 ± 0.0005


## Takeaways

A few things stood out once I had the numbers:

- Both LR models land around 0.99 on every metric. That lines up with what I saw in the EDA — spam here is driven by such strong individual words that even a simple linear model separates it easily.
- I expected TF-IDF to clearly beat the plain count baseline, but across 5 seeds they overlap within their standard deviation. So I can't really call TF-IDF an improvement — they're effectively tied.
- The more interesting split is precision. Since a false positive means a real email gets buried in spam, precision is what I'd actually optimize for here — and the baseline (0.981) edges out TF-IDF (0.975), which is also noticeably less stable (about 4x the variance). Ranking by F1 alone would have hidden this.
- Both models predict in ~1 µs per email on CPU, which is the bar DistilBERT has to clear in notebook 03. It does win on precision, but at roughly 8,000x the inference cost — so whether it's worth it really depends on the deployment.

> One note on honesty: I first measured on a single split (seed 42), then re-ran across multiple seeds because I wanted to know if the gaps were real or just noise. The comparison table reports the multi-seed mean ± std.